# Akan-BPE — 2A4 Llama BPB diagnostic & truncation-corrected eval

Helper notebook for **Phase 2A4** (`meta-llama/Llama-3.2-1B`). The 2A4 run was the only
ladder rung where the Akan fine-tune appeared to *lose* to the base on bits-per-byte
(mean-subword **0.8966** vs an unusually low base **0.7685**, report §11).

This notebook tests the hypothesis that the low Llama base BPB is a **truncation artifact**
of the metric, not real Twi-modeling skill:

- `_build_causal_example` truncates each text to `max_length-1` tokens, but
  `compute_model_bpb` divides the (truncated) NLL by the **full** UTF-8 byte count of the text.
- The eval texts are multi-sentence paragraphs, so longer-than-budget texts have their BPB
  **underestimated — worse for higher-fertility tokenizers** (they hit the token budget after
  fewer words). Llama's base tokenizer is the least efficient on Twi (~3.07 tok/word), so its
  base BPB is deflated the most.

**Plan:** (A) setup → (B) load eval + tokenizers → (C) quantify truncation →
(D) reproduce the 0.7685 artifact → (E) corrected **base** BPB (full byte coverage, no
adapters) → (F) retrain both Llama arms in-session → (G) corrected **experiment** BPB +
corrected improvement → (H) summary / verdict.

Runs top-to-bottom on a Kaggle single T4. `meta-llama/Llama-3.2-1B` and `google/gemma-3-1b-pt`
are gated — accept their licenses and provide an HF token (cell 3).

## A. Setup

In [ ]:
# 1. Clone repo (skip if already inside it)
import os
from pathlib import Path

os.environ["CUDA_VISIBLE_DEVICES"] = "0"
REPO = "https://github.com/attabeezy/akan-bpe.git"
REPO_NAME = "akan-bpe"

# Guard against triple-nesting on cell re-run: only clone+cd when we are not
# already sitting inside the repo root.
if Path.cwd().name != REPO_NAME:
    if not Path(REPO_NAME).is_dir():
        !git clone {REPO}
    %cd {REPO_NAME}

print(f"Working directory: {Path.cwd()}")

In [ ]:
# 2. Install dependencies
%pip install -q -e ".[dev,train]"
%pip install -q bitsandbytes sentencepiece pandas

In [ ]:
# Hugging Face authentication — Llama and Gemma are gated models.
# Accept the licenses at https://huggingface.co/meta-llama/Llama-3.2-1B and
# https://huggingface.co/google/gemma-3-1b-pt first, then run this cell and paste
# a token with access (or set the HF_TOKEN env var / Kaggle Secret).
from huggingface_hub import login
login()

In [ ]:
# 3. Confirm GPU is available
import torch

if not torch.cuda.is_available():
    raise RuntimeError(
        "No GPU detected. Go to Runtime / Settings -> Accelerator -> GPU T4, then re-run."
    )
print(f"GPU : {torch.cuda.get_device_name(0)}")
print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

In [ ]:
# 4. Download Akan datasets from HuggingFace Hub
# --tts-limit 50000 pins the pristine-Twi corpus to the same 45,000/2,500/2,500
# split used in the report (the default would pull ~188k rows and shift the split).
!python scripts/download.py --output-dir data --tts-limit 50000

In [ ]:
# 5. Train TTS tokenizer (if not already present)
from pathlib import Path

if not Path("models/tts_tokenizer.json").exists():
    print("TTS tokenizer not found - training now ...")
    !python scripts/train_bpe.py --inputs data/pristine_twi_train.jsonl --output models/tts_tokenizer.json --name tts
else:
    sz = Path("models/tts_tokenizer.json").stat().st_size / 1e6
    print(f"TTS tokenizer already present ({sz:.1f} MB), skipping.")

In [ ]:
# 6. Verify all required inputs exist
from pathlib import Path

required = {
    "TTS train data": Path("data/pristine_twi_train.jsonl"),
    "TTS test data": Path("data/pristine_twi_test.jsonl"),
    "TTS tokenizer": Path("models/tts_tokenizer.json"),
}
missing = [name for name, p in required.items() if not p.exists()]
if missing:
    raise FileNotFoundError(f"Missing required inputs: {missing}")
print("All inputs ready.")
for name, p in required.items():
    print(f"  {name}: {p}  ({p.stat().st_size / 1e6:.1f} MB)")

## B. Load eval texts + tokenizers

The full **2,500-row** pristine-Twi test set (the 2A4 run only scored a 512-row cap).
`load_texts` returns the untruncated texts; `load_experiment_tokenizer` loads the Akan TTS
tokenizer exactly as the runs do.

In [ ]:
import math
import numpy as np
import pandas as pd

from akan_bpe.model_integration import (
    load_texts,
    load_experiment_tokenizer,
    compute_model_bpb,        # original (truncating) BPB - used to reproduce the artifact
    build_text_dataset,       # original fixed-width dataset builder (truncates)
    _load_base_model_for_bpb, # quantized base-model loader
    _import_training_stack,   # torch + AutoModelForCausalLM + BitsAndBytesConfig + PeftModel
    _set_model_token_config,
)
from akan_bpe.metrics import count_utf8_bytes, bits_per_byte
from transformers import AutoTokenizer

# Shared runtime stack (use this torch everywhere so device handling matches the library).
stack = _import_training_stack()
torch = stack["torch"]
AutoModelForCausalLM = stack["AutoModelForCausalLM"]
BitsAndBytesConfig = stack["BitsAndBytesConfig"]
PeftModel = stack["PeftModel"]

MAX_LENGTH = 256   # the token budget used by the original runs (truncating path)
CHUNK_SIZE = 256   # full-coverage scoring chunk size (matches training context)

EVAL_FILE = Path("data/pristine_twi_test.jsonl")
AKAN_TOK_PATH = Path("models/tts_tokenizer.json")

eval_texts = load_texts(EVAL_FILE)                 # full 2,500 rows, untruncated
EVAL_BYTES = count_utf8_bytes(eval_texts)
print(f"Eval texts        : {len(eval_texts):,}")
print(f"Total UTF-8 bytes : {EVAL_BYTES:,}")

akan_tok = load_experiment_tokenizer(AKAN_TOK_PATH)

LLAMA_ID = "meta-llama/Llama-3.2-1B"
BASE_MODELS = {
    "Llama-3.2-1B": LLAMA_ID,
    "Qwen3-1.7B": "Qwen/Qwen3-1.7B",
    "gemma-3-1b-pt": "google/gemma-3-1b-pt",
}

# Base tokenizers for the truncation diagnosis. Tolerant: a gated/unavailable model is
# skipped rather than killing the notebook (Llama is the focus).
base_toks = {}
for name, mid in BASE_MODELS.items():
    try:
        base_toks[name] = AutoTokenizer.from_pretrained(mid)
    except Exception as exc:  # noqa: BLE001
        print(f"  [skip] base tokenizer {name} ({mid}): {exc}")
print("Loaded base tokenizers:", list(base_toks))

## C. Truncation diagnosis (no model load)

For each tokenizer over the **full** eval set: token-length distribution, the **fraction of
texts truncated** at `max_length=256`, and the **covered-byte fraction** — the share of each
text's bytes that the original (truncating) BPB actually scored. A low covered-byte fraction
means the original BPB divided real NLL by bytes it never scored, deflating it.

In [ ]:
def truncation_stats(tokenizer, texts, max_length):
    """Token-length distribution + truncation/coverage stats for one tokenizer."""
    budget = max_length - 1  # _build_causal_example keeps max_length-1 tokens, then EOS
    tok_lens = []
    covered_bytes = 0
    full_bytes = 0
    for t in texts:
        ids = tokenizer(t, add_special_tokens=False)["input_ids"]
        tok_lens.append(len(ids))
        fb = len(t.encode("utf-8"))
        full_bytes += fb
        if len(ids) > budget:
            cov_text = tokenizer.decode(ids[:budget])
            covered_bytes += min(len(cov_text.encode("utf-8")), fb)
        else:
            covered_bytes += fb
    a = np.array(tok_lens)
    return {
        "mean_tokens": round(float(a.mean()), 1),
        "median_tokens": int(np.median(a)),
        "p90_tokens": int(np.percentile(a, 90)),
        "max_tokens": int(a.max()),
        "pct_truncated": round(100.0 * float((a > budget).mean()), 1),
        "covered_byte_frac": round(covered_bytes / full_bytes, 4) if full_bytes else 0.0,
    }


rows = {"Akan-TTS": truncation_stats(akan_tok, eval_texts, MAX_LENGTH)}
for name, tok in base_toks.items():
    rows[name] = truncation_stats(tok, eval_texts, MAX_LENGTH)

diag = pd.DataFrame(rows).T
print("Truncation diagnosis at max_length =", MAX_LENGTH)
diag

### Full-coverage BPB helper

`compute_model_bpb_full` scores **100% of every text's bytes** via consecutive
non-overlapping token chunks (no truncation). It reuses the same causal-shift sum-NLL math as
`akan_bpe.model_integration.compute_model_bpb` and the same `count_utf8_bytes` / `bits_per_byte`
primitives, so it is consistent with the library metric — only the coverage differs. Because
every model scores all bytes of the same texts, base and experiment BPB stay comparable.

Non-overlapping chunks lose left-context at each chunk boundary — a small effect that is
symmetric across models; `chunk_size` is a parameter and we cross-check 256 vs 512 below.

In [ ]:
def compute_model_bpb_full(model, texts, tokenizer, torch, chunk_size=256):
    """Full byte-coverage bits-per-byte: no truncation, every byte scored."""
    cross_entropy = torch.nn.CrossEntropyLoss(reduction="sum")  # dense chunks: no -100 needed
    total_nll_nats = 0.0
    num_target_tokens = 0
    eos_id = tokenizer.eos_token_id
    model.eval()
    with torch.no_grad():
        for text in texts:
            ids = tokenizer(text, add_special_tokens=False)["input_ids"]
            if eos_id is not None:
                ids = ids + [eos_id]
            for start in range(0, len(ids), chunk_size):
                chunk = ids[start:start + chunk_size]
                if len(chunk) < 2:
                    continue  # need >=2 tokens for one prediction target
                input_ids = torch.tensor([chunk], device=model.device)
                outputs = model(input_ids=input_ids)
                shift_logits = outputs.logits[:, :-1, :]
                shift_labels = input_ids[:, 1:]
                loss = cross_entropy(
                    shift_logits.reshape(-1, shift_logits.size(-1)).float(),
                    shift_labels.reshape(-1),
                )
                total_nll_nats += float(loss.item())
                num_target_tokens += int(shift_labels.numel())
    total_bytes = count_utf8_bytes(texts)
    return {
        "bits_per_byte": bits_per_byte(total_nll_nats, total_bytes),
        "total_nll_bits": total_nll_nats / math.log(2),
        "total_bytes": total_bytes,
        "num_target_tokens": num_target_tokens,
    }


def free():
    """Collect garbage and clear the CUDA cache. Caller must `del` its own model
    reference *before* calling this (Python can only release the object once no name
    binds it)."""
    import gc
    gc.collect()
    torch.cuda.empty_cache()

## D. Reproduce the 0.7685 artifact

Score base Llama with the **original truncating** `compute_model_bpb` on the same capped config
as the run (first **512** texts, `max_length=256`). This should reproduce the reported base BPB
of ~**0.7685**, confirming the notebook faithfully replicates the pipeline before we correct it.

In [ ]:
llama_base_tok = base_toks.get("Llama-3.2-1B") or AutoTokenizer.from_pretrained(LLAMA_ID)
llama_base = _load_base_model_for_bpb(LLAMA_ID, AutoModelForCausalLM, torch, quantized=True)

# Original method, run config: 512 texts, max_length 256 (truncating), full bytes of those 512.
repro_texts = eval_texts[:512]
repro_ds = build_text_dataset(repro_texts, llama_base_tok, MAX_LENGTH)
repro = compute_model_bpb(llama_base, repro_ds, count_utf8_bytes(repro_texts), torch)
print(f"Reproduced base Llama BPB (512 rows, trunc max_length=256): {repro.bits_per_byte:.4f}")
print("Report  base Llama BPB (2A4, same capped config)          : 0.7685")

### Sanity check: full-coverage equals the library metric when nothing truncates

On texts that fit within the token budget for both tokenizers, no truncation happens, so
`compute_model_bpb_full` must equal `compute_model_bpb`. We assert this on a short-text subset
(base Llama), validating the new helper against the existing one.

In [ ]:
budget = MAX_LENGTH - 1
short_texts = [
    t for t in eval_texts[:400]
    if len(akan_tok(t, add_special_tokens=False)["input_ids"]) <= budget
    and len(llama_base_tok(t, add_special_tokens=False)["input_ids"]) <= budget
][:64]
print(f"Short non-truncated texts used for the check: {len(short_texts)}")

old_ds = build_text_dataset(short_texts, llama_base_tok, MAX_LENGTH)
old_bpb = compute_model_bpb(llama_base, old_ds, count_utf8_bytes(short_texts), torch).bits_per_byte
full_bpb = compute_model_bpb_full(llama_base, short_texts, llama_base_tok, torch, CHUNK_SIZE)["bits_per_byte"]
print(f"old (library)     : {old_bpb:.6f}")
print(f"full (this helper): {full_bpb:.6f}")
assert abs(old_bpb - full_bpb) < 1e-3, "full-coverage helper diverged from the library metric on non-truncated texts"
print("OK - helper matches the library metric when nothing truncates.")

## E. Corrected BASE BPB (full byte coverage, no adapters)

For each base model: the **original** truncating BPB on the full 2,500 rows vs the
**corrected** full-coverage BPB. The base-outlier question needs no adapters — just the base
model off the Hub. Headline check: does Llama's base BPB rise substantially from 0.7685?

In [ ]:
def score_base(name, model_id, model=None, tokenizer=None):
    own_model = model is None
    if tokenizer is None:
        tokenizer = base_toks.get(name) or AutoTokenizer.from_pretrained(model_id)
    if model is None:
        model = _load_base_model_for_bpb(model_id, AutoModelForCausalLM, torch, quantized=True)
    old_full = compute_model_bpb(
        model, build_text_dataset(eval_texts, tokenizer, MAX_LENGTH), EVAL_BYTES, torch
    ).bits_per_byte
    corrected = compute_model_bpb_full(model, eval_texts, tokenizer, torch, CHUNK_SIZE)["bits_per_byte"]
    if own_model:
        del model
        free()
    return {"old_trunc_full2500": round(old_full, 4), "corrected_full": round(corrected, 4),
            "delta": round(corrected - old_full, 4)}


base_bpb = {}
# Reuse the already-loaded Llama base (from cell D) to save a reload.
base_bpb["Llama-3.2-1B"] = score_base("Llama-3.2-1B", LLAMA_ID, model=llama_base, tokenizer=llama_base_tok)
del llama_base
free()

for name, mid in BASE_MODELS.items():
    if name == "Llama-3.2-1B":
        continue
    if name not in base_toks:
        print(f"  [skip] {name}: base tokenizer unavailable")
        continue
    try:
        base_bpb[name] = score_base(name, mid)
    except Exception as exc:  # noqa: BLE001
        print(f"  [skip] {name}: {exc}")

base_df = pd.DataFrame(base_bpb).T
print("Base BPB: original (truncating) vs corrected (full coverage), full 2,500 rows")
base_df

## F. Retrain the Llama Akan adapter in-session (both arms)

Reproduce the 2A4 adapters from code (seed 42) — no Kaggle retrieval. Same hyperparameters as
the 2A4 run; only `--embedding-init-mode` differs. Adapters are written to `*_helper` dirs so
they never collide with the committed run. (`meta-llama/Llama-3.2-1B` is already on the
`colab-qlora` allowlist.)

In [ ]:
# Arm A - random embedding init
!python scripts/model_integration.py \
    --experiment-id phase2a4_llama_3_2_1b_tts_helper \
    --model-id meta-llama/Llama-3.2-1B \
    --tokenizer-path models/tts_tokenizer.json \
    --train-file data/pristine_twi_train.jsonl \
    --eval-file data/pristine_twi_test.jsonl \
    --output-dir models/phase2a4_llama_3_2_1b_tts_helper \
    --results-output results/phase2a4_llama_3_2_1b_tts_helper.json \
    --device-mode colab-qlora \
    --embedding-init-mode random \
    --max-train-samples 4096 \
    --max-eval-samples 512 \
    --max-length 256 \
    --batch-size 1 \
    --grad-accum 16 \
    --epochs 1 \
    --learning-rate 2e-4 \
    --lora-r 16

In [ ]:
# Arm B - mean-of-subword embedding init
!python scripts/model_integration.py \
    --experiment-id phase2a4_llama_3_2_1b_tts_meansub_helper \
    --model-id meta-llama/Llama-3.2-1B \
    --tokenizer-path models/tts_tokenizer.json \
    --train-file data/pristine_twi_train.jsonl \
    --eval-file data/pristine_twi_test.jsonl \
    --output-dir models/phase2a4_llama_3_2_1b_tts_meansub_helper \
    --results-output results/phase2a4_llama_3_2_1b_tts_meansub_helper.json \
    --device-mode colab-qlora \
    --embedding-init-mode mean_subword \
    --max-train-samples 4096 \
    --max-eval-samples 512 \
    --max-length 256 \
    --batch-size 1 \
    --grad-accum 16 \
    --epochs 1 \
    --learning-rate 2e-4 \
    --lora-r 16

## G. Corrected EXPERIMENT BPB + corrected improvement

Reload each saved adapter following the same sequence as
`verify_saved_qwen_artifacts` (resize embeddings, then `PeftModel.from_pretrained`, with the
resized `embed_tokens`/`lm_head` restored via `modules_to_save`). Score the fine-tuned model
with full byte coverage using **its own Akan tokenizer** (reloaded from the adapter dir), then
compute the corrected `base - experiment` against the corrected Llama base BPB from cell E.

In [ ]:
def reload_adapter(output_dir, model_id):
    """Mirror akan_bpe.model_integration.verify_saved_qwen_artifacts reload sequence."""
    tok = AutoTokenizer.from_pretrained(output_dir)
    if tok.pad_token is None and tok.eos_token is not None:
        tok.pad_token = tok.eos_token
    qc = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_quant_type="nf4",
        bnb_4bit_use_double_quant=True,
        bnb_4bit_compute_dtype=torch.float16,
    )
    base = AutoModelForCausalLM.from_pretrained(model_id, quantization_config=qc, dtype="auto")
    base.config.tie_word_embeddings = False
    _set_model_token_config(base, tok)
    base.resize_token_embeddings(len(tok), pad_to_multiple_of=64)
    model = PeftModel.from_pretrained(base, output_dir)
    _set_model_token_config(model, tok)
    model.eval()
    return model, tok


ARMS = {
    "random": "models/phase2a4_llama_3_2_1b_tts_helper",
    "mean_subword": "models/phase2a4_llama_3_2_1b_tts_meansub_helper",
}
base_corrected = base_bpb["Llama-3.2-1B"]["corrected_full"]

exp_bpb = {}
for arm, out_dir in ARMS.items():
    model, tok = reload_adapter(out_dir, LLAMA_ID)
    corrected = compute_model_bpb_full(model, eval_texts, tok, torch, CHUNK_SIZE)["bits_per_byte"]
    exp_bpb[arm] = {
        "experiment_corrected": round(corrected, 4),
        "base_corrected": round(base_corrected, 4),
        "improvement_corrected": round(base_corrected - corrected, 4),
    }
    del model
    free()

exp_df = pd.DataFrame(exp_bpb).T
print("Corrected experiment BPB (full coverage) and corrected improvement vs base")
exp_df

## H. Summary — original vs truncation-corrected

Side-by-side of the reported (512-row, truncating) 2A4 numbers and the corrected
(full 2,500-row, full-coverage) numbers, with the verdict on whether Llama flips positive.

In [ ]:
# Reported 2A4 numbers (report section 11; 512-row capped, truncating BPB).
REPORTED = {
    "base":         0.7685,
    "random":       {"experiment": 1.0357, "improvement": -0.2672},
    "mean_subword": {"experiment": 0.8966, "improvement": -0.1281},
}

summary = []
for arm in ("random", "mean_subword"):
    summary.append({
        "arm": arm,
        "base_reported": REPORTED["base"],
        "base_corrected": exp_bpb[arm]["base_corrected"],
        "exp_reported": REPORTED[arm]["experiment"],
        "exp_corrected": exp_bpb[arm]["experiment_corrected"],
        "improv_reported": REPORTED[arm]["improvement"],
        "improv_corrected": exp_bpb[arm]["improvement_corrected"],
    })
summary_df = pd.DataFrame(summary).set_index("arm")
print("2A4 Llama: reported (truncating, 512 rows) vs corrected (full coverage, 2,500 rows)")
print(summary_df.to_string())

flipped = exp_bpb["mean_subword"]["improvement_corrected"] > 0
print()
print(f"mean_subword corrected improvement: {exp_bpb['mean_subword']['improvement_corrected']:+.4f}"
      f"  ->  {'FLIPS POSITIVE (Akan wins after correction)' if flipped else 'still negative'}")
print()
print("Note: every base tokenizer has higher fertility than the Akan tokenizer, so the same")
print("truncation correction should raise the Akan advantage at 2A1-2A3 too. If confirmed, the")
print("real fix is in the library (build_text_dataset / compute_model_bpb -> full coverage) plus")
print("a ladder re-run and a report.md update.")

### Optional cross-check: chunk_size 256 vs 512

Re-scoring corrected base Llama at `chunk_size=512` should give the same qualitative
conclusion (base rises well above 0.7685; sign of the corrected 2A4 improvement unchanged).
Run this only if you want the robustness check — it reloads the Llama base once more.

In [ ]:
llama_base2 = _load_base_model_for_bpb(LLAMA_ID, AutoModelForCausalLM, torch, quantized=True)
for cs in (256, 512):
    r = compute_model_bpb_full(llama_base2, eval_texts, llama_base_tok, torch, chunk_size=cs)
    print(f"corrected base Llama BPB @ chunk_size={cs}: {r['bits_per_byte']:.4f}")
del llama_base2
free()